In [1]:
# Sel ini membuat dataset baru untuk Tugas Mandiri Pertemuan 4 dan mengunggahnya ke HDFS
import numpy as np
import pandas as pd

np.random.seed(99)
n = 1000
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga", "Olahraga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo", "Kebumen"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-09-01", "2026-09-30", freq="D")

data = {
    "order_id": [f"ORD-{3000 + i}" for i in range(n)],
    "tanggal": np.random.choice(tanggal_range, size=n).astype(str),
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 12, size=n),
    "harga_satuan": np.random.choice([20000, 45000, 60000, 90000, 125000, 200000, 350000], size=n),
    "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    "rating": np.random.choice([1, 2, 3, 4, 5, np.nan], size=n, p=[0.03, 0.02, 0.10, 0.30, 0.35, 0.20]),
}
df_tugas4 = pd.DataFrame(data)
df_tugas4.to_csv("transaksi_september_2026.csv", index=False)
print(f"Dataset dibuat: {df_tugas4.shape[0]} baris")

# Mengunggah ke HDFS
!hdfs dfs -mkdir -p /user/mahasiswa/tugas4
!hdfs dfs -put -f transaksi_september_2026.csv /user/mahasiswa/tugas4/
print("Berhasil diunggah ke HDFS: /user/mahasiswa/tugas4/transaksi_september_2026.csv")

Dataset dibuat: 1000 baris
Berhasil diunggah ke HDFS: /user/mahasiswa/tugas4/transaksi_september_2026.csv


In [ ]:
## Persiapan SparkSession

Pada tahap ini dibuat SparkSession sebagai titik awal untuk menjalankan proses pengolahan data menggunakan PySpark.

In [7]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Tugas4-PySpark") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("SparkSession berhasil dibuat!")
print("Versi Spark:", spark.version)

SparkSession berhasil dibuat!
Versi Spark: 3.5.6


In [ ]:
## A. Membaca Dataset dari HDFS

Pada tahap ini dataset transaksi dibaca langsung dari HDFS menggunakan PySpark DataFrame. Setelah data berhasil dibaca, ditampilkan schema, jumlah baris, dan 10 data pertama untuk mengetahui struktur awal dataset.

In [8]:
df = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas4/transaksi_september_2026.csv",
    header=True,
    inferSchema=True
)

print("Schema:")
df.printSchema()

print("Jumlah baris:", df.count())

print("10 data pertama:")
df.show(10)

Schema:
root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)

Jumlah baris: 1000
10 data pertama:
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kec

In [ ]:
## B. Menangani Missing Value

Pada tahap ini dilakukan pengecekan nilai kosong pada kolom rating. Nilai kosong kemudian diisi dengan 0 menggunakan na.fill() agar seluruh data transaksi tetap dipertahankan dan tidak ada baris yang terhapus.

In [9]:
from pyspark.sql.functions import col

jumlah_kosong = df.filter(
    col("rating").isNull()
).count()

print("Jumlah rating kosong:", jumlah_kosong)

df = df.na.fill({"rating": 0})

print(
    "Jumlah rating kosong setelah ditangani:",
    df.filter(col("rating").isNull()).count()
)

Jumlah rating kosong: 204
Jumlah rating kosong setelah ditangani: 0


In [ ]:
## C. Menambahkan Kolom Total Pendapatan dan Tier Transaksi

Pada tahap ini dibuat kolom total_pendapatan dengan mengalikan unit_terjual dan harga_satuan. Selanjutnya dibuat kolom tier_transaksi dengan ketentuan "Besar" jika total pendapatan lebih dari 500000 dan "Kecil" jika tidak memenuhi kondisi tersebut.

In [10]:
from pyspark.sql.functions import when

df = df.withColumn(
    "total_pendapatan",
    col("unit_terjual") * col("harga_satuan")
)

df = df.withColumn(
    "tier_transaksi",
    when(col("total_pendapatan") > 500000, "Besar")
    .otherwise("Kecil")
)

df.select(
    "order_id",
    "unit_terjual",
    "harga_satuan",
    "total_pendapatan",
    "tier_transaksi"
).show(10)## D1. Kategori dengan Total Pendapatan Tertinggi

Data dikelompokkan berdasarkan kategori dan dihitung total pendapatannya. Hasilnya diurutkan dari total pendapatan terbesar sehingga kategori pada urutan pertama merupakan kategori dengan pendapatan tertinggi.

+--------+------------+------------+----------------+--------------+
|order_id|unit_terjual|harga_satuan|total_pendapatan|tier_transaksi|
+--------+------------+------------+----------------+--------------+
|ORD-3000|           3|       90000|          270000|         Kecil|
|ORD-3001|           3|      200000|          600000|         Besar|
|ORD-3002|           8|       60000|          480000|         Kecil|
|ORD-3003|           6|      350000|         2100000|         Besar|
|ORD-3004|          10|       60000|          600000|         Besar|
|ORD-3005|           5|       20000|          100000|         Kecil|
|ORD-3006|           2|       20000|           40000|         Kecil|
|ORD-3007|           8|       90000|          720000|         Besar|
|ORD-3008|           7|       20000|          140000|         Kecil|
|ORD-3009|          10|       90000|          900000|         Besar|
+--------+------------+------------+----------------+--------------+
only showing top 10 rows



In [ ]:
## D1. Kategori dengan Total Pendapatan Tertinggi

Data dikelompokkan berdasarkan kategori dan dihitung total pendapatannya. Hasilnya diurutkan dari total pendapatan terbesar sehingga kategori pada urutan pertama merupakan kategori dengan pendapatan tertinggi.

In [11]:
from pyspark.sql.functions import sum as spark_sum

hasil_kategori = df.groupBy("kategori").agg(
    spark_sum("total_pendapatan").alias("total_pendapatan")
).orderBy(
    col("total_pendapatan").desc()
)

hasil_kategori.show()

+--------------------+----------------+
|            kategori|total_pendapatan|
+--------------------+----------------+
|        Rumah Tangga|       138665000|
|   Makanan & Minuman|       131890000|
|Kesehatan & Kecan...|       128595000|
|            Olahraga|       126650000|
|             Fashion|       124075000|
|          Elektronik|       110295000|
+--------------------+----------------+



In [ ]:
## D2. Kota dengan Transaksi Besar Terbanyak

Data difilter untuk mengambil transaksi dengan tier_transaksi "Besar". Kemudian data dikelompokkan berdasarkan kota dan dihitung jumlah transaksinya untuk mengetahui kota dengan transaksi Besar terbanyak.

In [12]:
hasil_kota = df.filter(
    col("tier_transaksi") == "Besar"
).groupBy(
    "kota"
).count().orderBy(
    col("count").desc()
)

hasil_kota.show()

+----------+-----+
|      kota|count|
+----------+-----+
|      Solo|   92|
|  Magelang|   78|
|   Kebumen|   78|
|Yogyakarta|   75|
| Purworejo|   66|
|  Semarang|   65|
+----------+-----+



In [ ]:
## D3. Rata-rata Rating per Metode Pembayaran

Data dikelompokkan berdasarkan metode pembayaran, kemudian dihitung rata-rata rating untuk setiap metode pembayaran. Nilai kosong pada rating telah ditangani sebelumnya sehingga perhitungan dapat dilakukan pada data yang sudah diproses.

In [13]:
from pyspark.sql.functions import avg

hasil_rating = df.groupBy(
    "metode_pembayaran"
).agg(
    avg("rating").alias("rata_rata_rating")
).orderBy(
    col("rata_rata_rating").desc()
)

hasil_rating.show()

+-----------------+------------------+
|metode_pembayaran|  rata_rata_rating|
+-----------------+------------------+
|              COD|3.3745019920318726|
|    Transfer Bank|3.3399209486166006|
|         E-Wallet|             3.292|
|     Kartu Kredit|3.1910569105691056|
+-----------------+------------------+



In [ ]:
## E. Menyimpan Hasil Pengolahan ke HDFS

Pada tahap ini DataFrame yang sudah diproses disimpan kembali ke HDFS dalam format CSV. Spark menyimpan hasil dalam beberapa file partisi seperti part-00000 karena proses pengolahan data dilakukan secara terdistribusi.

In [14]:
output_path = "hdfs://localhost:9000/user/mahasiswa/tugas4/hasil_transaksi"

df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(output_path)

print("Data berhasil disimpan ke HDFS.")

[Stage 21:>                                                         (0 + 1) / 1]

Data berhasil disimpan ke HDFS.


In [ ]:
### Verifikasi Hasil Penyimpanan

Pada tahap ini dilakukan pengecekan isi folder hasil pada HDFS untuk memastikan data hasil pengolahan sudah berhasil disimpan.

In [15]:
!hdfs dfs -ls /user/mahasiswa/tugas4/hasil_transaksi

Found 2 items
-rw-r--r--   3 asih2 supergroup          0 2026-09-10 09:12 /user/mahasiswa/tugas4/hasil_transaksi/_SUCCESS
-rw-r--r--   3 asih2 supergroup      97296 2026-09-10 09:12 /user/mahasiswa/tugas4/hasil_transaksi/part-00000-cdb21d3e-3f7c-4d32-9b77-691a58b015c0-c000.csv


In [ ]:
### Membaca Kembali Data Hasil

Data hasil pengolahan dibaca kembali dari HDFS untuk memastikan file hasil penyimpanan dapat digunakan dan berisi data yang telah diproses.

In [16]:
df_hasil = spark.read.csv(
    output_path,
    header=True,
    inferSchema=True
)

df_hasil.show(10)

+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|total_pendapatan|tier_transaksi|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|          270000|         Kecil|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|          600000|         Besar|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|           8|       60000|         E-Wallet|   3.0|          480000|         Kecil|
|ORD-3003|2026-09-09 00:00:00|   Makanan & Minuman|  Semarang|           6|      350000|    Transfer Bank|   4.0|         21